# Stage 0 - download Zenodo 4266616 and build timelines (CPU runtime is enough)

Run cells top to bottom. If any cell errors, read the full error message; do not edit the loader to be less strict.

In [ ]:
# one-time: put the package on Drive and install it. Upload the folder stresstwin_p/ (the one with pyproject.toml)
# to  MyDrive/stresstwin_p  first.
from google.colab import drive; drive.mount('/content/drive')
%pip -q install -e /content/drive/MyDrive/stresstwin_p
import os; WORK='/content/drive/MyDrive/stresstwin_work'; os.makedirs(WORK, exist_ok=True)

In [ ]:
# 1) download (CC-BY-4.0). File name, size and md5 are taken from https://zenodo.org/records/4266616
import hashlib, subprocess, os
RAW = f"{WORK}/raw"; os.makedirs(RAW, exist_ok=True)
zp = f"{RAW}/MentalHealthSubreddits2017.zip"
if not os.path.exists(zp):
    subprocess.run(["wget","-q","--show-progress","-O",zp,"https://zenodo.org/records/4266616/files/MentalHealthSubreddits2017.zip?download=1"],check=True)
h = hashlib.md5(open(zp,"rb").read()).hexdigest()
print("md5", h, "OK" if h=="64a61e5883f2dbe80fbfdf173df984f9" else "MISMATCH - do not continue")
assert h=="64a61e5883f2dbe80fbfdf173df984f9"
subprocess.run(["unzip","-q","-o",zp,"-d",RAW],check=True)
for r,_,fs in os.walk(RAW):
    for f in sorted(fs):
        if f.endswith(".pkl"): print(os.path.join(r,f), round(os.path.getsize(os.path.join(r,f))/1e6,1),"MB")

In [ ]:
# 2) secret salt for author hashing: created once, stored on Drive, never committed or shared
import secrets, pathlib
sf = pathlib.Path(WORK)/'salt.bin'
if not sf.exists(): sf.write_bytes(secrets.token_bytes(32))
print('salt ok', sf.stat().st_size, 'bytes')

In [ ]:
# 3) build tables. A SchemaError prints the keys actually found; compare them with the expected schema.
!python -m stresstwin_p.timeline_builder --folder {WORK}/raw --salt-file {WORK}/salt.bin --out {WORK}/tables

In [ ]:
import pandas as pd
o = pd.read_csv(f'{WORK}/tables/observations.csv'); print(o.groupby('community').author.nunique())
g = o.groupby('author').agg(n=('t_days','size'), span=('t_days', lambda s: s.max()-s.min()))
print((g.n>=5).sum(), 'authors with >=5 obs;', ((g.n>=5)&(g.span>=14)).sum(), 'also spanning >=14 days')
# DECISION RULE: fewer than ~40 eligible authors in a community => it cannot be a held-out community. Report it, do not hide it.